# MAPbI$_3$ under humidity: statistical sampling with randomly inserted H$_2$O


In this notebook we build a **2×2×2 supercell** of orthorhombic MAPbI$_3$ and generate an **ensemble of structures**
containing water molecules in random positions and random orientations.

## Why generate many structures?

A single manually chosen geometry would not be representative of what happens in the real material after exposure
to humidity. In practice, water can approach the organic cations in many different local arrangements.
The goal here is therefore to create **M different initial structures** with **N H$_2$O molecules each**, so that
we obtain a simple **statistical sampling** of plausible local environments.

These structures are only **initial guesses**. In a realistic workflow they should then be:

1. **geometry optimized**, and
2. compared in terms of their **energies** and structural features.

## Exercise

Construct a 2×2×2 supercell of the orthorhombic phase of MAPbI$_3$ using the structural parameters reported in  
the Supplementary Information of: https://doi.org/10.1039/C4CC09944C

Then generate several atomistic models such that:

- each model contains **N H$_2$O molecules**,
- the **oxygen atom** of each H$_2$O molecule is placed at a distance between **1.5 Å and 2.5 Å** from a nitrogen atom of methylammonium,
- every atom of the newly inserted H$_2$O molecule stays at a distance of at least **1.5 Å** from all atoms already present in the structure.

We keep the advanced AiiDAlab structure visualizer in the notebook so that the generated structures can be inspected interactively.


In [ ]:
%load_ext aiida
%aiida

## Imports

We use:

- **ASE** to construct and manipulate crystal structures,
- **SciPy** to generate proper random 3D rotations,
- **AiiDAlab widgets** to inspect the structures interactively.


In [1]:
import numpy as np
from datetime import datetime
from ase.io import write
from ase.spacegroup import crystal
from ase import neighborlist
from ase.build import molecule
from aiida import orm

from scipy.spatial.transform import Rotation

import ipywidgets as ipw
import aiidalab_widgets_base as awb
from surfaces_tools.widgets import editors


<IPython.core.display.Javascript object>

## Interactive structure viewer

The widget below is used throughout the notebook to inspect the pristine supercell and the generated samples.


In [2]:
structure_selector = awb.StructureManagerWidget(
    importers=[
        awb.StructureUploadWidget(title="Import from computer"),
        awb.StructureBrowserWidget(title="AiiDA database"),
        awb.SmilesWidget(title="From SMILES"),
    ],
    editors=[
        awb.BasicStructureEditor(title="Edit structure"),
        awb.BasicCellEditor(),
        editors.InsertStructureWidget(title="Insert molecule"),
    ],
    storable=True,
    node_class="StructureData",
)


## Step 1: Build the orthorhombic MAPbI$_3$ unit cell

The lattice parameters and fractional coordinates are taken from the reference above.


In [3]:
a = 8.86574
b = 12.6293
c = 8.57684
alpha = 90
beta = 90
gamma = 90

MAPbI3 = crystal(
    symbols="PbI2NCH4",  # some labels in the source table are slightly misleading
    basis=[
        (0.5,    0.0,   0.0),
        (0.4842, 0.25, -0.0562),
        (0.1886, 0.0147, 0.1844),
        (0.9421, 0.75,  0.0297),
        (0.9372, 0.25,  0.0575),
        (0.9372, 0.25,  0.1874),
        (0.8661, 0.1701, 0.0290),
        (0.1275, 0.1891,-0.0085),
        (0.9543, 0.750, 0.1459),
    ],
    spacegroup=62,
    cellpar=[a, b, c, alpha, beta, gamma],
)


## Step 2: Create the 2×2×2 supercell

This is the host structure into which water molecules will be inserted.


In [4]:
supercell_no_h2o = MAPbI3.repeat((2, 2, 2))
structure_selector.structure = supercell_no_h2o
supercell_no_h2o


/home/jovyan/.local/lib/python3.9/site-packages/aiidalab_widgets_base/viewers.py:737: DeprecationWarning: dict interface is deprecated. Use attribute interface instead
  self.cell_spacegroup.value = f"Spacegroup: {symmetry_dataset['international']} (No.{symmetry_dataset['number']})"


Atoms(symbols='C32H192I96N32Pb32', pbc=True, cell=[17.73148, 25.2586, 17.15368], _aiidalab_viewer_representation_default=..., spacegroup_kinds=...)

## Step 3: Identify the nitrogen atoms

Each water molecule will be inserted near a randomly chosen nitrogen atom belonging to a methylammonium cation.

We also identify the hydrogen atoms bound to nitrogen. This is not strictly required for the insertion algorithm,
but it is often useful when inspecting the local environment.


In [5]:
cutoffs = neighborlist.natural_cutoffs(supercell_no_h2o)
nl = neighborlist.NeighborList(cutoffs, self_interaction=False, bothways=True)
nl.update(supercell_no_h2o)

all_N = [atom.index for atom in supercell_no_h2o if atom.symbol == "N"]
all_H_of_N = [
    index
    for N in all_N
    for index in nl.get_neighbors(N)[0]
    if supercell_no_h2o[index].symbol == "H"
]
all_nh3 = all_N + all_H_of_N

print(f"Number of nitrogen atoms in the 2x2x2 supercell: {len(all_N)}")
print(f"Number of H atoms bonded to N: {len(all_H_of_N)}")


Number of nitrogen atoms in the 2x2x2 supercell: 32
Number of H atoms bonded to N: 96


## Step 4: Define the sampling parameters

- `NH2O`: number of water molecules per sample
- `nsamples`: number of different structures to generate
- `dmin_shell`, `dmax_shell`: allowed distance range between the **oxygen** atom of water and a nitrogen atom
- `min_distance`: minimum allowed distance between **any atom of the trial water molecule** and the atoms already present in the structure
- `max_attempts_per_water`: safety limit for the rejection-sampling loop
- `seed`: random seed for reproducibility


In [6]:
NH2O = 10
nsamples = 5

dmin_shell = 1.5
dmax_shell = 2.5
min_distance = 1.5

max_attempts_per_water = 1000
seed = 4732


## Step 5: Helper functions


### uniform random rotation in 3D
using `scipy.spatial.transform.Rotation.random()`.

### Strategy used below

For each H$_2$O molecule:

1. choose a random nitrogen atom,
2. choose a random direction in 3D,
3. choose a random O–N distance between `dmin_shell` and `dmax_shell`,
4. apply a random 3D rotation to the water molecule,
5. accept the placement only if all distances from the new water atoms to the atoms already present in the structure are at least `min_distance`.


In [7]:
def random_unit_vector(rng):
    """Return a random unit vector sampled uniformly on the sphere."""
    vec = rng.normal(size=3)
    return vec / np.linalg.norm(vec)


def make_centered_h2o():
    """
    Build an H2O molecule and translate it so that the oxygen atom sits at the origin.
    This makes rotations and translations easier to apply.
    """
    h2o = molecule("H2O")
    h2o.translate(-h2o.positions[0])  # oxygen is atom 0 in ASE's H2O
    return h2o


def propose_h2o_near_random_n(base_h2o, structure, n_indices, dmin_shell, dmax_shell, rng):
    """
    Create a trial H2O molecule with:
    - oxygen placed at a random distance from a random nitrogen atom,
    - a random orientation in 3D.
    """
    trial_h2o = base_h2o.copy()

    # Choose a random N atom in the pristine MAPbI3 host.
    n_index = int(rng.choice(n_indices))
    n_position = structure.positions[n_index]

    # Choose a random point in the spherical shell around that N atom.
    direction = random_unit_vector(rng)
    distance = rng.uniform(dmin_shell, dmax_shell)
    oxygen_position = n_position + distance * direction

    # Apply a proper random 3D rotation to the water molecule.
    rotation = Rotation.random(random_state=rng)
    trial_h2o.positions = rotation.apply(trial_h2o.positions)

    # Translate so that the oxygen atom reaches the target position.
    trial_h2o.translate(oxygen_position)

    return trial_h2o, n_index


def placement_is_valid(structure, trial_h2o, min_distance):
    """
    Return True if every atom of trial_h2o is at least min_distance away from every atom already
    present in structure. Distances are computed with the minimum-image convention.
    """
    natoms_existing = len(structure)
    natoms_trial = len(trial_h2o)

    combined = structure + trial_h2o

    for i_trial in range(natoms_existing, natoms_existing + natoms_trial):
        for j_existing in range(natoms_existing):
            if combined.get_distance(i_trial, j_existing, mic=True, vector=False) < min_distance:
                return False

    return True


def generate_one_sample(supercell_no_h2o, n_indices, NH2O, dmin_shell, dmax_shell, min_distance, rng, max_attempts_per_water=1000, verbose=True):
    """
    Generate one sample containing NH2O water molecules.
    """
    structure = supercell_no_h2o.copy()
    base_h2o = make_centered_h2o()

    for ih2o in range(NH2O):
        inserted = False

        for attempt in range(1, max_attempts_per_water + 1):
            trial_h2o, n_index = propose_h2o_near_random_n(
                base_h2o=base_h2o,
                structure=structure,
                n_indices=n_indices,
                dmin_shell=dmin_shell,
                dmax_shell=dmax_shell,
                rng=rng,
            )

            if placement_is_valid(structure, trial_h2o, min_distance=min_distance):
                structure += trial_h2o
                inserted = True
                if verbose:
                    print(
                        f"  inserted H2O {ih2o + 1:2d}/{NH2O} "
                        f"after {attempt:3d} attempts near N index {n_index}"
                    )
                break

        if not inserted:
            raise RuntimeError(
                f"Could not place H2O number {ih2o + 1} after {max_attempts_per_water} attempts. "
                "Try decreasing NH2O, increasing the shell size, or relaxing the minimum-distance cutoff."
            )

    structure.wrap()
    return structure


def generate_samples(supercell_no_h2o, n_indices, nsamples, NH2O, dmin_shell, dmax_shell, min_distance, seed=7, max_attempts_per_water=1000, verbose=True):
    """
    Generate nsamples different structures.
    """
    rng = np.random.default_rng(seed)
    samples = []

    for isample in range(nsamples):
        if verbose:
            print(f"Creating sample {isample + 1}/{nsamples}")
        sample = generate_one_sample(
            supercell_no_h2o=supercell_no_h2o,
            n_indices=n_indices,
            NH2O=NH2O,
            dmin_shell=dmin_shell,
            dmax_shell=dmax_shell,
            min_distance=min_distance,
            rng=rng,
            max_attempts_per_water=max_attempts_per_water,
            verbose=verbose,
        )
        samples.append(sample)

    return samples


## Step 6: Generate the statistical ensemble

This cell creates `nsamples` different structures.  
Each structure contains `NH2O` water molecules placed according to the rules above.


In [8]:
samples = generate_samples(
    supercell_no_h2o=supercell_no_h2o,
    n_indices=all_N,
    nsamples=nsamples,
    NH2O=NH2O,
    dmin_shell=dmin_shell,
    dmax_shell=dmax_shell,
    min_distance=min_distance,
    seed=seed,
    max_attempts_per_water=max_attempts_per_water,
    verbose=True,
)


Creating sample 1/5
  inserted H2O  1/10 after   7 attempts near N index 355
  inserted H2O  2/10 after   7 attempts near N index 306
  inserted H2O  3/10 after  37 attempts near N index 307
  inserted H2O  4/10 after   7 attempts near N index 64
  inserted H2O  5/10 after  16 attempts near N index 352
  inserted H2O  6/10 after  26 attempts near N index 17
  inserted H2O  7/10 after   6 attempts near N index 211
  inserted H2O  8/10 after  10 attempts near N index 258
  inserted H2O  9/10 after   2 attempts near N index 66
  inserted H2O 10/10 after   8 attempts near N index 307
Creating sample 2/5
  inserted H2O  1/10 after  11 attempts near N index 65
  inserted H2O  2/10 after   8 attempts near N index 259
  inserted H2O  3/10 after   5 attempts near N index 163
  inserted H2O  4/10 after  31 attempts near N index 161
  inserted H2O  5/10 after  41 attempts near N index 211
  inserted H2O  6/10 after   7 attempts near N index 17
  inserted H2O  7/10 after   2 attempts near N index 

## Step 7: Inspect the generated samples interactively

Use the slider to send one of the generated structures to the AiiDAlab viewer.


In [9]:
sample_slider = ipw.IntSlider(
    value=0,
    min=0,
    max=len(samples) - 1,
    step=1,
    description="Sample:",
    continuous_update=False,
)

def show_sample(sample_index):
    structure_selector.structure = samples[sample_index]

ipw.interact(show_sample, sample_index=sample_slider);


interactive(children=(IntSlider(value=0, continuous_update=False, description='Sample:', max=4), Output()), _d…

In [10]:
display(structure_selector)

StructureManagerWidget(children=(Accordion(children=(Tab(children=(StructureUploadWidget(children=(FileUpload(…

## Step 8: Save the generated structures

The structures are written as  `xyz` files containing  atomic positions, cell, and periodic boundary conditions.

You can later optimize these structures, for examle with AiiDAlab (**requires computational resources beyond the availability in this course**), and compare their total energies to obtain a more realistic picture of the effect of humidity on MAPbI$_3$.

In the next cell, we can also store the structures in the AiiDA database as `StructureData` to make them directly available in AiiDAlab

In [ ]:
Store2AiiDA = False
Write2Disk = False
for i, sample in enumerate(samples):
    filename = f"MAPbI3_2x2x2_{NH2O}H2O_sample_{i:02d}.xyz"
    label = f"MAPbI3_2x2x2_{NH2O}H2O_sample_{i:02d}_{datetime.now().strftime('%Y%m%d%H%M%S%f')}"
    if Store2AiiDA:
        aiidageo=orm.StructureData(ase=sample)
        aiidageo.label = label
        aiidageo.store()
        print(f"stored in AiiDA {label} with PK {aiidageo.pk}")
    if Write2Disk:
        write(filename, sample)

print(f"Saved {len(samples)} structures.")


## Final remarks

This notebook generates a **statistical ensemble of chemically plausible initial geometries** rather than a single deterministic structure.

That is the right spirit for this problem: humidity does not produce one unique local arrangement, but a distribution of possible local environments.  
The next  step is therefore to:

- relax them (we will see how this could be done with AiiDAlab and inspect the simulation with `verdi node show`, `verdi node attributes`, `verdi process list`, and with `verdi calcjob gotocomputer`, and with `verdi calcjob inputcat`),
- compare their energies,
- and analyze the spread of structural motifs that remain favorable after optimization (see lecture 15th April).
